## 0. Google Colab Setup

Run this notebook in Google Colab with the teaching folder stored in Google Drive. Keep `config.py`, `Network.py`, `plot_utils.py`, and the required raw NPZ files inside the Drive folder before starting.

The next cell installs Cartopy, mounts Google Drive, and moves into the Drive-based teaching folder so data, checkpoints, plots, and prediction files are stored persistently.


In [ ]:
# Install Cartopy for map plotting in Colab.
!pip install -q cartopy

# Mount Google Drive. The notebook uses Drive for data and saved outputs.
from google.colab import drive
drive.mount("/content/drive")

# Teaching folder in Google Drive. Adjust only if your Drive folder has a different name.
%cd /content/drive/MyDrive/dl_downscaling_teaching


# WRF U-Net Downscaling: Preprocessing, Training, Testing, and Inference

This project develops a compact deep-learning workflow for downscaling Weather Research and Forecasting (WRF) model outputs over the Pearl River Delta (PRD) region. The objective is to learn a statistical mapping from coarse-resolution atmospheric fields to fine-resolution fields using paired WRF simulations.

The notebook focuses on deterministic residual downscaling with a U-Net. For each time step, the model receives coarse WRF weather variables together with static geographic information, such as terrain height and land-sea mask. It then predicts the correction required to reconstruct the corresponding fine-resolution field.

**Workflow Overview**

In [ ]:
from IPython import display
display.Image("dl_model_struct.png")

## 1. Learning Goals

By the end of this notebook, you should be able to:

- explain why atmospheric downscaling is needed when coarse models cannot resolve local PRD-scale weather features
- compare coarse and fine WRF variables before training
- describe the preprocessing pipeline: grid matching, precipitation transformation, static channels, residual targets, and normalization
- train a compact residual U-Net using processed tensors
- reconstruct physical fine-resolution fields from normalized model outputs
- visualize and save coarse, predicted, and fine fields for later assessment

The central modelling assumption is that the coarse WRF field already contains the broad atmospheric pattern. The U-Net is therefore trained to estimate the remaining correction needed to make the coarse field closer to the fine-resolution reference.


## 2. Set Up Paths

Run the Colab setup cell above first so the notebook is inside the Google Drive `dl_downscaling_teaching` directory. The file `config.py` serves as the central configuration file for data paths, output paths, variable names, normalization constants, and training settings.

Because the working directory is in Google Drive, checkpoints, diagnostic plots, and inference files are saved persistently under the Drive-based `outputs/` folder.


In [ ]:
import os

import config as cfg

# Create the output folder before saving checkpoints, plots, or predictions.
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

# Print the main paths so the file inputs and outputs are explicit.
print("Teaching directory:    ", cfg.PROJECT_DIR)
print("Data directory:        ", cfg.DATA_DIR)
print("Output directory:      ", cfg.OUTPUT_DIR)

## 3. Import Libraries

This workflow uses NumPy for NPZ arrays, PyTorch for model training, torchvision transforms for resizing and normalization, and Matplotlib-based helper functions for plotting weather maps. The local files `Network.py`, `plot_utils.py`, and `config.py` provide the model implementation, visualization utilities, and configuration values.

Gridded atmospheric fields can be represented in an image-like form: each weather variable is treated as a channel, while the latitude-longitude grid forms the spatial plane. This representation makes convolutional neural networks, including U-Net architectures, suitable for the downscaling task.


In [ ]:
import numpy as np
import torch
from torch.utils.data import DataLoader
from torchvision.transforms import InterpolationMode, Normalize, Resize
import matplotlib.pyplot as plt
from IPython.display import clear_output, display
from tqdm.auto import tqdm

from Network import UNet
from plot_utils import (
    DEFAULT_UNIT_LABELS,
    plot_case_study_comparison,
    plot_density_qq_grid,
    plot_loss_curve,
    plot_weather_grid,
)

# Use GPU when Colab provides one; otherwise use CPU.
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

# Keep variable lists in one place so later cells use the same channel order.
WEATHER_VARIABLES = list(cfg.IN_OUT_PARAMS)
STATIC_VARIABLES = list(cfg.CONSTANT_PARAMS)

print("Using device:", DEVICE)
print("Weather variables:", WEATHER_VARIABLES)
print("Static variables: ", STATIC_VARIABLES)

## 4. Data Helpers

These helpers keep only the repeated work that would otherwise distract from the preprocessing steps. The raw weather arrays are structured like a stack of images: time first, then weather variable, followed by the latitude-longitude grid.

Only three helpers are used:

- `get_channel_stats(stats)`: returns means and standard deviations in model channel order.
- `load_weather_npz(path)`: reads one raw NPZ and selects the six model weather channels.
- `residual_to_fine(...)`: converts predicted normalized residuals back to physical units and adds them to the matched coarse field.

**Important precipitation naming note**

The raw NPZ files store rainfall as `ACC_6H_PRECIP`, which means accumulated 6-hour precipitation in millimeters. The loader keeps these raw millimeter values unchanged. Section 8 transforms the `ACC_6H_PRECIP` values in-place with `log(precipitation + 0.001)`, so this channel is not raw rainfall while the model is training.


In [ ]:
def get_channel_stats(stats):
    means = [stats["mean"][name] for name in WEATHER_VARIABLES]
    stds = [stats["std"][name] for name in WEATHER_VARIABLES]
    return means, stds


def load_weather_npz(path):
    with np.load(path, allow_pickle=False) as npz_file:
        raw_data = npz_file["data"]
        raw_names = list(npz_file["variable_names"])
        time = npz_file["time"]
        lat = npz_file["lat"]
        lon = npz_file["lon"]

    selected_channels = []
    for name in WEATHER_VARIABLES:
        raw_name = name
        channel = raw_data[:, raw_names.index(raw_name)].astype(np.float32)
        selected_channels.append(channel)

    data = np.stack(selected_channels, axis=1).astype(np.float32)
    return data, time, lat, lon


def residual_to_fine(residual_norm, coarse, mean, std):
    # Convert normalized residuals back to physical units, then add coarse fields.
    mean = torch.tensor(mean, dtype=residual_norm.dtype).view(1, -1, 1, 1)
    std = torch.tensor(std, dtype=residual_norm.dtype).view(1, -1, 1, 1)
    mean = mean.to(residual_norm.device)
    std = std.to(residual_norm.device)
    residual = residual_norm * std + mean
    return coarse + residual

## 5. Choose Raw Files And Normalization Statistics

The notebook uses predefined train and test NPZ files containing paired coarse-resolution and fine-resolution WRF outputs over the PRD downscaling domain. The coarse fields provide the model inputs, while the fine fields provide the supervised learning targets.

The split is chronological:

- training years: 2021 and 2022
- testing year: 2023

This matters in scientific machine learning because nearby weather samples can be correlated. Testing on a later year gives a clearer check of whether the model generalizes beyond the period used for training.

The statistics in `config.py` were precomputed from the training years only and then manually stored in the configuration file. They are reused for both training and testing. This avoids using information from the 2023 test set during preprocessing. Two sets of statistics are required:

- raw-field statistics for normalizing coarse weather inputs
- residual statistics for normalizing fine-minus-coarse targets

The next cell loads the train and test raw NPZ pairs once. Later preprocessing and plotting cells reuse these in-memory arrays instead of opening the same NPZ files again.


In [ ]:
# Load channel-wise normalization values from config.py.
raw_mean, raw_std = get_channel_stats(cfg.RAW_STATS)
residual_mean, residual_std = get_channel_stats(cfg.RESIDUAL_STATS)

# Load each raw NPZ once. Precipitation is still in millimeters at this stage.
train_coarse_raw, train_time, train_coarse_lat, train_coarse_lon = load_weather_npz(cfg.TRAIN_COARSE_RAW_NPZ_PATH)
train_fine_raw, train_fine_time, train_fine_lat, train_fine_lon = load_weather_npz(cfg.TRAIN_FINE_RAW_NPZ_PATH)
test_coarse_raw, test_time, test_coarse_lat, test_coarse_lon = load_weather_npz(cfg.TEST_COARSE_RAW_NPZ_PATH)
test_fine_raw, test_fine_time, test_fine_lat, test_fine_lon = load_weather_npz(cfg.TEST_FINE_RAW_NPZ_PATH)

print("Training years:      2021, 2022")
print("Testing year:        2023")
print("Train coarse file:  ", cfg.TRAIN_COARSE_RAW_NPZ_PATH)
print("Train fine file:    ", cfg.TRAIN_FINE_RAW_NPZ_PATH)
print("Test coarse file:   ", cfg.TEST_COARSE_RAW_NPZ_PATH)
print("Test fine file:     ", cfg.TEST_FINE_RAW_NPZ_PATH)
print("Train coarse shape: ", train_coarse_raw.shape)
print("Train fine shape:   ", train_fine_raw.shape)
print("Test coarse shape:  ", test_coarse_raw.shape)
print("Test fine shape:    ", test_fine_raw.shape)
print("Epochs:             ", cfg.NUM_EPOCHS)
print("Batch size:         ", cfg.BATCH_SIZE)
print("Learning rate:      ", cfg.LEARNING_RATE)


## 6. Plot All Coarse And Fine Variables

Initial data inspection is an essential step in scientific machine learning. The model uses several atmospheric variables because precipitation is influenced by surrounding thermodynamic and dynamical conditions, not by rainfall alone.

This notebook uses six weather channels:

- `T2`: 2-m air temperature
- `TD2`: 2-m dew point temperature
- `MSLP`: mean sea level pressure
- `U10`: 10-m east-west wind component
- `V10`: 10-m north-south wind component
- `ACC_6H_PRECIP`: 6-hour accumulated precipitation, log-transformed before training

The next cell creates a figure with the coarse variables on the first row and fine variables on the second row using the already-loaded training arrays. The comparison should be used to assess large-scale consistency between paired fields and the additional spatial detail present in the fine-resolution fields.


In [ ]:
# Pick one timestep for all comparison plots.
SAMPLE_TIME_INDEX = 300
SAMPLE_VARIABLE = "T2"
variable_index = WEATHER_VARIABLES.index(SAMPLE_VARIABLE)

# Build one plot panel for every coarse weather variable.
items = []
for i in range(len(WEATHER_VARIABLES)):
    name = WEATHER_VARIABLES[i]
    item = {"field": train_coarse_raw[SAMPLE_TIME_INDEX,i], "variable": name, "title": "Coarse " + name}
    items.append(item)

# Add one plot panel for every fine weather variable.
for i in range(len(WEATHER_VARIABLES)):
    name = WEATHER_VARIABLES[i]
    item = {"field": train_fine_raw[SAMPLE_TIME_INDEX,i], "variable": name, "title": "Fine " + name, "lat": train_fine_lat, "lon": train_fine_lon}
    items.append(item)

# Plot all panels in two rows: coarse first, fine second.
fig = plot_weather_grid(items, ncols=len(WEATHER_VARIABLES), figsize_per_panel=(3.0, 2.8))
display(fig)
plt.close(fig)

## 7. Preprocessing Step 1: Match Coarse And Fine Grids

A U-Net requires inputs and targets to share the same spatial dimensions. Both coarse and fine fields are therefore resized to a common `48 x 48` analysis grid before residual targets are calculated.

The `48 x 48` grid is a teaching analysis grid used by this workflow. It is not meant to claim that the original fine WRF data naturally has exactly `48 x 48` grid cells. The raw coarse and raw fine files start with different grid sizes, then both are placed onto the same model grid so that `fine - coarse` can be computed cell by cell.

Nearest-neighbor resizing is used for the coarse field so that the original coarse-grid values are not smoothed into new artificial values. Bilinear resizing is used for the fine field because it is already the higher-resolution reference and can be smoothly placed on the common grid.

The next cell resizes every already-loaded train and test timestep onto the shared model grid, then plots one before-and-after comparison.


In [ ]:
# Build reusable resizers for all raw NPZ pairs.
coarse_resize = Resize(cfg.HIGH_RES_SHP, interpolation=InterpolationMode.NEAREST, antialias=False)

fine_resize = Resize(cfg.HIGH_RES_SHP, interpolation=InterpolationMode.BILINEAR, antialias=False)

# Convert and resize the full train and test datasets once.
train_coarse = coarse_resize(torch.tensor(train_coarse_raw, dtype=torch.float32))
train_fine = fine_resize(torch.tensor(train_fine_raw, dtype=torch.float32))
test_coarse = coarse_resize(torch.tensor(test_coarse_raw, dtype=torch.float32))
test_fine = fine_resize(torch.tensor(test_fine_raw, dtype=torch.float32))

# Plot the grid-matching effect for one variable and timestep.
items = [
    {"field": train_coarse_raw[SAMPLE_TIME_INDEX, variable_index], "variable": SAMPLE_VARIABLE, "title": "Raw coarse", "lat": train_coarse_lat, "lon": train_coarse_lon},
    {"field": train_coarse[SAMPLE_TIME_INDEX, variable_index], "variable": SAMPLE_VARIABLE, "title": "Matched coarse"},
    {"field": train_fine_raw[SAMPLE_TIME_INDEX, variable_index], "variable": SAMPLE_VARIABLE, "title": "Raw fine", "lat": train_fine_lat, "lon": train_fine_lon},
    {"field": train_fine[SAMPLE_TIME_INDEX, variable_index], "variable": SAMPLE_VARIABLE, "title": "Matched fine"},
]
fig = plot_weather_grid(items, ncols=4)
display(fig)
plt.close(fig)

## 8. Preprocessing Step 2: Transform Precipitation

Precipitation is challenging for neural networks because it is non-negative, frequently zero, and strongly skewed toward a small number of high-rainfall values. A logarithmic transformation is applied before normalization to reduce this skewness.

The transformation is defined as:

$$
\hat{x} = \ln(x + 0.001)
$$

where $x$ denotes the original precipitation value in millimeters, and $\hat{x}$ denotes the transformed precipitation value used by the model. The constant `0.001` prevents the logarithm of zero. This transformation compresses very large rainfall values while preserving differences among small rainfall values.

For this channel, the model does not directly learn raw millimeters of rain. It learns residuals in log-precipitation space. The plotting helpers can convert precipitation back to approximate millimeters for visual display, but the training target is still based on the transformed variable.

The transformation is applied in this preprocessing step, after grid matching and before normalization.


In [ ]:
# Transform precipitation in-place after grid matching and before normalization.
precip_index = WEATHER_VARIABLES.index("ACC_6H_PRECIP")
raw_precip = train_coarse[SAMPLE_TIME_INDEX,precip_index].clone()

for data in [train_coarse, train_fine, test_coarse, test_fine]:
    data[:, precip_index] = torch.log(torch.clamp(data[:,precip_index], min=0.0) + 0.001)

log_precip = train_coarse[SAMPLE_TIME_INDEX,precip_index]

items = [
    {"field": raw_precip, "variable": "ACC_6H_PRECIP", "title": "Original precipitation", "colorbar_label": "mm"},
    {"field": log_precip, "variable": "LOG_PRECIP", "title": "Log precipitation", "colorbar_label": "log(mm + 0.001)"}
]
fig = plot_weather_grid(items, ncols=2)
display(fig)
plt.close(fig)

## 9. Preprocessing Step 3: Build Static Channels

Static fields do not vary with time, but they provide important geographic information for downscaling. Terrain height and land-sea contrast can influence local wind, temperature, and precipitation patterns, especially in coastal and topographically complex regions such as the PRD.

This notebook appends two static channels to every time step:

- `LSM`: land-sea mask
- `HGT`: terrain height

Resized with bilinear interpolation and normalize by z-score

The model therefore receives eight input channels in total: 6 weather channels + 2 static channels. The next cell builds these static channels once and reuses them for both train and test arrays.


In [ ]:
# Static fields do not change with time, so load them once.
static_channels = []
with np.load(cfg.STATIC_NPZ_PATH, allow_pickle=False) as static_npz:
    for variable in STATIC_VARIABLES:
        channel = torch.tensor(static_npz[variable], dtype=torch.float32)[None,None]

        # Resize it with bilinear interpolation.
        static_resize = Resize(cfg.HIGH_RES_SHP, interpolation=InterpolationMode.BILINEAR, antialias=False)

        channel = static_resize(channel)[0,0]

        # Terrain height is standardized so its scale is easier for the model.
        if variable != "LSM":
            channel = (channel - channel.mean()) / (channel.std() + 1e-6)
        static_channels.append(channel)

static_grid = torch.stack(static_channels, dim=0)

# Plot the static channels after resizing and standardization.
items = []
for i in range(len(STATIC_VARIABLES)):
    name = STATIC_VARIABLES[i]
    item = {"field": static_grid[i], "variable": name, "title": name}
    items.append(item)
fig = plot_weather_grid(items, ncols=len(STATIC_VARIABLES))
display(fig)
plt.close(fig)

# Repeat static channels so every timestep receives the same static information.
train_static = static_grid.expand(train_coarse.shape[0], -1, -1, -1)
test_static = static_grid.expand(test_coarse.shape[0], -1, -1, -1)

## 10. Preprocessing Step 4: Build Residual Targets

The model is trained to predict the difference between the matched fine field and the matched coarse field, rather than the full fine field directly. This target is referred to as the residual.

```text
target residual = fine field - coarse field
predicted fine  = coarse field + predicted residual
```

Residual learning is appropriate here because the coarse WRF field already contains much of the large-scale atmospheric structure. The U-Net can therefore focus on estimating local corrections associated with spatial detail, terrain-related effects, and systematic differences between the two grids.

In plain language, the model is answering: "What correction should I add to the coarse field to make it closer to the fine field?"

After prediction, the estimated residual is added back to the coarse field to reconstruct the downscaled fine field. The next cell builds the residual targets for the already matched train and test tensors.


In [ ]:
# The model learns fine minus coarse instead of the whole fine field.
train_residual = train_fine - train_coarse
test_residual = test_fine - test_coarse

# Plot matched coarse, residual target, and matched fine for one variable.
items = [
    {"field": train_coarse[SAMPLE_TIME_INDEX,variable_index], "variable": SAMPLE_VARIABLE, "title": "Coarse"},
    {"field": train_fine[SAMPLE_TIME_INDEX,variable_index], "variable": SAMPLE_VARIABLE, "title": "Fine"},
    {"field": train_residual[SAMPLE_TIME_INDEX,variable_index], "variable": SAMPLE_VARIABLE, "title": "Residual (Fine - Coarse)"}
]
fig = plot_weather_grid(items, ncols=3)
display(fig)
plt.close(fig)

## 11. Preprocessing Step 5: Normalize And Build Model Arrays

Neural networks generally train more reliably when different input and output channels have comparable numerical scales. This notebook uses z-score normalization:

$$
z = \frac{x - \mu}{\sigma}
$$

where $x$ is the original value, $\mu$ is the training-set mean, $\sigma$ is the training-set standard deviation, and $z$ is the normalized value. For example, if a temperature value is 300 K, the training-set mean is 295 K, and the training-set standard deviation is 5 K, the normalized value is `(300 - 295) / 5 = 1`.

The model sees and predicts the following tensors:

```text
model input  = normalized coarse weather channels + static LSM/HGT channels
model output = normalized residual 
final field  = coarse weather channels + de-normalized predicted residual
```

The next code cell builds the model-ready train and test tensors from the already matched arrays.


In [ ]:
# Normalize the coarse weather fields before giving them to the model.
input_normalizer = Normalize(mean=raw_mean, std=raw_std)
train_weather_inputs = input_normalizer(train_coarse)
test_weather_inputs = input_normalizer(test_coarse)

# Normalize residual targets so all output channels have comparable scale.
target_normalizer = Normalize(mean=residual_mean, std=residual_std)
train_targets = target_normalizer(train_residual)
test_targets = target_normalizer(test_residual)

# Final model input is weather channels plus static channels.
train_inputs = torch.cat([train_weather_inputs, train_static], dim=1)
test_inputs = torch.cat([test_weather_inputs, test_static], dim=1)

# The saved fields are on the resized 48 x 48 analysis grid.
lat = np.linspace(cfg.DL_DOMAIN["max_lat"], cfg.DL_DOMAIN["min_lat"], cfg.HIGH_RES_SHP[0])
lon = np.linspace(cfg.DL_DOMAIN["min_lon"], cfg.DL_DOMAIN["max_lon"], cfg.HIGH_RES_SHP[1])

# Plot physical and normalized values side by side for one variable.
items = [
    {"field": train_coarse[SAMPLE_TIME_INDEX,variable_index], "variable": SAMPLE_VARIABLE, "title": "Physical coarse"},
    {"field": train_inputs[SAMPLE_TIME_INDEX,variable_index], "variable": "NORMALIZED", "title": "Normalized coarse", "colorbar_label": "standardized value"},
    {"field": train_residual[SAMPLE_TIME_INDEX,variable_index], "variable": SAMPLE_VARIABLE, "title": "Physical residual"},
    {"field": train_targets[SAMPLE_TIME_INDEX,variable_index], "variable": "NORMALIZED", "title": "Normalized residual", "colorbar_label": "standardized value"},
]
fig = plot_weather_grid(items, ncols=4)
display(fig)
plt.close(fig)

## 12. Verify Processed Tensor Structure

Verification immediately after preprocessing is important for reproducible data processing. The next cell checks the in-memory tensor shapes and data types.

For this notebook, each input sample should contain 8 channels on the `48 x 48` analysis grid. Each target sample should contain 6 weather-variable residual channels on the same grid.

Expected tensor meanings:

```text
inputs:  (N, 8, 48, 48) = 6 normalized weather channels + 2 normalized static channels
targets: (N, 6, 48, 48) = normalized residuals for 6 weather variables
coarse:  (N, 6, 48, 48) = matched coarse fields in physical units
fine:    (N, 6, 48, 48) = matched fine reference fields in physical units
```

Here `N` is the number of time steps. If these dimensions do not match, the most common causes are missing raw NPZ files, a changed variable list, or an inconsistent preprocessing step.


In [ ]:
print("Inputs: ", tuple(train_inputs.shape), train_inputs.dtype)
print("Targets:", tuple(train_targets.shape), train_targets.dtype)
print("Coarse: ", tuple(train_coarse.shape), train_coarse.dtype)
print("Fine:   ", tuple(train_fine.shape), train_fine.dtype)
print("Time:   ", train_time.shape, train_time.dtype)
print("Lat:    ", lat.shape, lat.dtype)
print("Lon:    ", lon.shape, lon.dtype)

## 13. PyTorch Dataset For Processed Tensors

PyTorch models train from `Dataset` objects. This small class wraps the processed tensors already built in memory and returns one time step at a time.

A custom PyTorch dataset is usually made by subclassing `torch.utils.data.Dataset` and defining three methods:

- `__init__`: runs once when the dataset is created. Use it to store the data arrays, file paths, labels, coordinates, or any metadata the dataset will need later.
- `__len__`: returns the number of samples in the dataset. PyTorch uses this to know how many items are available.
- `__getitem__`: returns one sample for a given index. A `DataLoader` repeatedly calls this method, then stacks the returned samples into mini-batches for training.

In this notebook, one sample corresponds to one WRF time step, so `__len__` returns the number of time steps and `__getitem__` returns the tensors for one selected time index.

Each sample is a dictionary containing:

- `inputs`: what the U-Net sees
- `targets`: the normalized residual the U-Net should predict
- `coarse`: the matched coarse field used later for reconstruction
- `fine`: the matched fine field used later for comparison

Keeping `coarse` and `fine` in the dataset makes the training data useful for both optimization and visualization.


In [ ]:
class ProcessedWRFDataset(torch.utils.data.Dataset):
    def __init__(self, inputs, targets, coarse, fine, time, lat, lon):
        # Store processed tensors directly; no intermediate NPZ file is needed.
        self.inputs = inputs
        self.targets = targets
        self.coarse = coarse
        self.fine = fine
        self.time = time
        self.lat = lat
        self.lon = lon

    def __len__(self):
        # Number of samples equals the number of timesteps.
        return self.inputs.shape[0]

    def __getitem__(self, index):
        # Return one timestep as a dictionary for the training loop.
        return {
            "inputs": self.inputs[index],
            "targets": self.targets[index],
            "coarse": self.coarse[index],
            "fine": self.fine[index],
        }

## 14. Create Train And Test DataLoaders

A `DataLoader` groups individual time steps into mini-batches. Mini-batching is faster than training on one time step at a time and gives the optimizer a more stable estimate of the gradient.

The training loader uses `shuffle=True` so the model does not see samples in chronological order every epoch. The test loader keeps `shuffle=False` because predictions should remain aligned with their original time metadata.

Each mini-batch contains several `48 x 48` weather examples at once. The batch size is controlled by `config.py`.


In [ ]:
# Wrap the processed tensors directly; no processed NPZ files are saved or reloaded.
train_dataset = ProcessedWRFDataset(train_inputs, train_targets, train_coarse, train_fine, train_time, lat, lon)
test_dataset = ProcessedWRFDataset(test_inputs, test_targets, test_coarse, test_fine, test_time, lat, lon)

# DataLoaders handle batching during training and testing.
train_loader = DataLoader(train_dataset, batch_size=cfg.BATCH_SIZE, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=0)

# Inspect one sample to verify the tensor shapes.
first_sample = train_dataset[0]
print("Train samples:      ", len(train_dataset))
print("Test samples:       ", len(test_dataset))
print("Input tensor shape: ", first_sample["inputs"].shape)
print("Target tensor shape:", first_sample["targets"].shape)

## 15. Build The Model And Loss

The U-Net is an encoder-decoder convolutional network. The encoder compresses spatial information into deeper feature maps, and the decoder upsamples those features back to the original grid. Skip connections pass high-resolution information from encoder layers to matching decoder layers, which helps preserve spatial structure.

You do not need to understand every line of `Network.py` to follow this teaching demo. That file contains a flexible U-Net implementation adapted from a more advanced diffusion-model codebase. In this notebook we use it as a compact residual-prediction network:

```text
Input:  8 channels  = 6 normalized coarse weather channels + LSM + HGT
Body:   encoder-decoder CNN with skip connections
Output: 6 channels  = normalized residuals for the 6 weather variables
```

The model predicts 6 normalized residual channels from 8 input channels. The training objective is mean squared error (MSE):

$$
MSE = \frac{1}{n}\sum_{i=1}^{n}(\hat{y}_i - y_i)^2
$$

where $\hat{y}_i$ is the model prediction at one grid point and channel, $y_i$ is the target value, and $n$ is the total number of compared values in the batch. Squaring the error gives larger penalties to larger prediction errors. The U-Net used here is intentionally compact so that it can be trained in a notebook environment while retaining the main encoder-decoder learning structure.


In [ ]:
# The input includes weather channels plus static channels.
num_input_channels = len(WEATHER_VARIABLES) + len(STATIC_VARIABLES)
num_output_channels = len(WEATHER_VARIABLES)

# Build a plain U-Net: no class labels and no time input.
model = UNet(
    img_resolution=cfg.HIGH_RES_SHP,
    in_channels=num_input_channels,
    out_channels=num_output_channels,
    **cfg.MODEL_KWARGS
)
model = model.to(DEVICE)

# Mean squared error compares predicted residuals with target residuals.
loss_fn = torch.nn.MSELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.LEARNING_RATE)

print("Input channels: ", num_input_channels)
print("Output channels:", num_output_channels)
print("Model is ready on", DEVICE)

## 16. Train And Test Functions

The training function performs the standard PyTorch optimization loop:

1. move a mini-batch to the selected device
2. predict normalized residuals
3. compute MSE against target residuals
4. backpropagate the loss
5. update model weights

The test function uses the same loss calculation but disables gradient tracking. Testing measures how well the model performs on held-out data without changing the trained parameters.

In plain language, every training step adjusts the U-Net so its predicted residual becomes closer to the true residual.


In [ ]:
def run_train_epoch(model, dataloader, optimizer):
    # Training mode enables gradient updates.
    model.train()
    losses = []

    for batch in tqdm(dataloader, desc="Train", leave=False):
        inputs = batch["inputs"].to(DEVICE)
        targets = batch["targets"].to(DEVICE)

        # Predict normalized residuals from normalized coarse inputs.
        prediction = model(inputs)
        loss = loss_fn(prediction, targets)

        # Standard PyTorch training step.
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        losses.append(loss.item())

    return float(np.mean(losses))


def run_test_epoch(model, dataloader):
    # Evaluation mode avoids training-only behavior.
    model.eval()
    losses = []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Test", leave=False):
            inputs = batch["inputs"].to(DEVICE)
            targets = batch["targets"].to(DEVICE)

            prediction = model(inputs)
            loss = loss_fn(prediction, targets)
            losses.append(loss.item())

    return float(np.mean(losses))

## 17. Run Training

Training repeats the train and test loops for several epochs. An epoch means that the model has processed every training sample once.

The checkpoint with the lowest test loss is saved, rather than simply retaining the final epoch. This is important because a model can continue improving on the training set while its performance on testing set begins to degrade.

The loss curve is the first diagnostic plot. A typical successful run should show decreasing training loss, while the test loss should remain broadly consistent with the training trend rather than diverging strongly.


In [ ]:
# Keep loss history so we can plot progress after each epoch.
train_losses = []
test_losses = []
best_test_loss = float("inf")

for epoch in range(1, cfg.NUM_EPOCHS + 1):
    train_loss = run_train_epoch(model, train_loader, optimizer)
    test_loss = run_test_epoch(model, test_loader)

    train_losses.append(train_loss)
    test_losses.append(test_loss)

    # Save the best model according to test loss.
    if test_loss < best_test_loss:
        best_test_loss = test_loss
        torch.save(model.state_dict(), cfg.BEST_CHECKPOINT_PATH)

    # Refresh the loss plot inside the notebook output.
    fig = plot_loss_curve(train_losses, test_losses, num_epochs=cfg.NUM_EPOCHS, title="Training Curve")
    fig.savefig(cfg.LOSS_CURVE_PATH, dpi=300, bbox_inches="tight")
    clear_output(wait=True)
    display(fig)
    plt.close(fig)

    # Print the full loss table after refreshing the plot.
    for i in range(len(train_losses)):
        completed_epoch = i + 1
        train_value = train_losses[i]
        test_value = test_losses[i]
        print(f"Epoch {completed_epoch} train loss = {train_value:.4f} test loss = {test_value:.4f}")

print("Best checkpoint saved to: ", cfg.BEST_CHECKPOINT_PATH)

## 18. Reload The Best Checkpoint For Inference

Before generating final prediction files, reload the checkpoint with the best test loss. This ensures inference uses the best model observed during training, even if the last epoch was not the best epoch.

A checkpoint stores learned model weights, not the raw data. The model architecture must therefore be built in the same way before loading the saved weights.


In [ ]:
# Reload the best checkpoint before running inference.
model.load_state_dict(torch.load(cfg.BEST_CHECKPOINT_PATH, map_location=DEVICE))
model.eval()

print("Loaded checkpoint for inference:", cfg.BEST_CHECKPOINT_PATH)

## 19. Predict The Test Dataset

Inference means applying the trained model without updating weights. For each 2023 test batch, the notebook predicts normalized residuals, converts them back to physical units or transformed-variable units, and adds them back to the matched coarse fields.

The final outputs are saved as three NPZ files so they can be compared outside the training loop:

- coarse test fields
- predicted fine fields
- fine reference fields

The model works internally with log-transformed precipitation, but the final saved NPZ files are written in the same core structure as the raw NPZ files. They contain `data`, `variable_names`, `time`, `lat`, and `lon`, and the precipitation channel is converted back to `ACC_6H_PRECIP` in millimeters.

The loader is not shuffled so every saved prediction remains aligned with the original 2023 test time index.


In [ ]:
def predict_dataset(model, dataset, batch_size):
    # Use a non-shuffled DataLoader so outputs stay in dataset order.
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0)

    coarse_batches = []
    fine_batches = []
    predicted_batches = []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Inference", leave=False):
            inputs = batch["inputs"].to(DEVICE)

            # Predict normalized residuals, then reconstruct physical fine fields.
            residual_prediction = model(inputs).cpu()
            predicted_fine = residual_to_fine(
                residual_prediction,
                batch["coarse"],
                residual_mean,
                residual_std,
            )

            coarse_batches.append(batch["coarse"])
            fine_batches.append(batch["fine"])
            predicted_batches.append(predicted_fine)

    # Join all mini-batches into full arrays.
    coarse = torch.cat(coarse_batches, dim=0).numpy()
    fine = torch.cat(fine_batches, dim=0).numpy()
    predicted = torch.cat(predicted_batches, dim=0).numpy()
    return coarse, fine, predicted


# Run inference on the test dataset and save predictions.
coarse_output, fine_output, predicted_output = predict_dataset(model, test_dataset, cfg.BATCH_SIZE)

# Save coarse, predicted, and fine fields using the raw NPZ-style structure.
COARSE_TEST_PATH = f"{cfg.OUTPUT_DIR}/wrf_test_coarse.npz"
PREDICTED_TEST_PATH = f"{cfg.OUTPUT_DIR}/wrf_test_predicted.npz"
FINE_TEST_PATH = f"{cfg.OUTPUT_DIR}/wrf_test_fine.npz"

def convert_model_channels_to_raw(data):
    # Internal precipitation is log(mm + 0.001); raw-style output uses mm.
    raw_data = data.copy()
    precip_index = WEATHER_VARIABLES.index("ACC_6H_PRECIP")
    raw_data[:, precip_index] = np.maximum(np.exp(raw_data[:, precip_index]) - 0.001, 0.0)
    return raw_data.astype(np.float32)


def save_weather_npz(path, data, time, lat, lon):
    np.savez_compressed(path, data=convert_model_channels_to_raw(data), time=time, lat=lat, lon=lon, variable_names=WEATHER_VARIABLES)


save_weather_npz(COARSE_TEST_PATH, coarse_output, test_dataset.time, test_dataset.lat, test_dataset.lon)
save_weather_npz(PREDICTED_TEST_PATH, predicted_output, test_dataset.time, test_dataset.lat, test_dataset.lon)
save_weather_npz(FINE_TEST_PATH, fine_output, test_dataset.time, test_dataset.lat, test_dataset.lon)

print("Coarse shape:        ", coarse_output.shape)
print("Fine shape:          ", fine_output.shape)
print("Predicted shape:     ", predicted_output.shape)
print("Saved variables:     ", WEATHER_VARIABLES)
print("Saved coarse file:   ", COARSE_TEST_PATH)
print("Saved predicted file:", PREDICTED_TEST_PATH)
print("Saved fine file:     ", FINE_TEST_PATH)

## 20. Visualize One Saved Prediction

This final figure reads from the raw-style NPZ files created during inference and displays coarse, predicted, and fine fields in a common layout.

When reviewing the figure, focus on whether the U-Net adds meaningful local structure while preserving the large-scale weather pattern inherited from the coarse input. That is the core goal of residual downscaling in this notebook.


In [ ]:
# Display one prediction example from the saved raw-style NPZ files.
TIME_IDX = 300
with np.load(COARSE_TEST_PATH, allow_pickle=False) as coarse_npz:
    saved_coarse = coarse_npz["data"]
    saved_time = coarse_npz["time"]
    saved_lat = coarse_npz["lat"]
    saved_lon = coarse_npz["lon"]
    saved_variables = list(coarse_npz["variable_names"])

with np.load(PREDICTED_TEST_PATH, allow_pickle=False) as predicted_npz:
    saved_predicted = predicted_npz["data"]

with np.load(FINE_TEST_PATH, allow_pickle=False) as fine_npz:
    saved_fine = fine_npz["data"]

rows = [("Coarse", saved_coarse[TIME_IDX]), ("Predicted", saved_predicted[TIME_IDX]), ("Fine", saved_fine[TIME_IDX])]

fig = plot_weather_grid(rows, ncols=len(saved_variables), figsize_per_panel=(3.3, 2.7), lat=saved_lat, lon=saved_lon, weather_variables=saved_variables)
display(fig)
plt.close(fig)

## 21. MAE And RMSE Error Table

The error table compares both the original coarse field and the U-Net prediction against the fine WRF reference. This makes it clear whether the DL downscaling improves over the coarse input for each variable.

Each row is calculated over the whole test domain and all test times for one model output variable.

In [ ]:
def make_error_metrics_table(coarse, fine, predicted, variable_names):
    """Create a whole-domain MAE/RMSE verification table for saved DL predictions."""
    import pandas as pd

    rows = []
    for channel_index, variable in enumerate(variable_names):
        coarse_values = np.asarray(coarse[:, channel_index], dtype=np.float64)
        fine_values = np.asarray(fine[:, channel_index], dtype=np.float64)
        predicted_values = np.asarray(predicted[:, channel_index], dtype=np.float64)

        coarse_error = coarse_values - fine_values
        predicted_error = predicted_values - fine_values
        rows.append([
            variable,
            np.nanmean(coarse_error),
            np.nanmean(predicted_error),
            np.nanmean(np.abs(coarse_error)),
            np.nanmean(np.abs(predicted_error)),
            np.sqrt(np.nanmean(coarse_error ** 2)),
            np.sqrt(np.nanmean(predicted_error ** 2)),
        ])

    columns = ["PARAM", "coarse_ME", "pred_ME", "coarse_MAE", "pred_MAE", "coarse_RMSE", "pred_RMSE"]
    return pd.DataFrame(rows, columns=columns)


In [ ]:
# Generate the MAE/RMSE verification table.
error_table = make_error_metrics_table(
    coarse=saved_coarse,
    fine=saved_fine,
    predicted=saved_predicted,
    variable_names=saved_variables,
)

ERROR_TABLE_PATH = f"{cfg.OUTPUT_DIR}/unet_error_metrics.csv"
error_table.to_csv(ERROR_TABLE_PATH, index=False)

display(error_table.round(3))
print("Saved error table:", ERROR_TABLE_PATH)


## 22. Combined Density And Q-Q Verification Plots

The combined density and Q-Q plot checks the full distribution of model predictions against the fine WRF reference. Each panel contains one output variable. The hexbin density shows where most grid-point comparisons fall, while the green Q-Q curve highlights distributional bias across quantiles.

A prediction distribution close to the reference should cluster near the red `y=x` line and have a Q-Q curve that stays close to that line.

In [ ]:
# Save and display all density/QQ plots in one 3-column figure.
fig = plot_density_qq_grid(
    fine=saved_fine,
    predicted=saved_predicted,
    variable_names=saved_variables,
    unit_labels=DEFAULT_UNIT_LABELS,
    ncols=3,
)

DENSITY_QQ_PATH = f"{cfg.OUTPUT_DIR}/density_scatter_QQ_all_variables.png"
fig.savefig(DENSITY_QQ_PATH, dpi=300, bbox_inches="tight")
display(fig)
plt.close(fig)

print("Saved density/QQ figure:", DENSITY_QQ_PATH)


## 23. Case Study Weather Map Comparison

A single case-study map is useful for checking whether the downscaled output preserves the synoptic pattern from the coarse WRF input while adding local detail closer to the fine WRF reference.

This final plot follows the reference case-study script: precipitation is shaded, mean sea-level pressure is contoured, and 10-m wind is shown with barbs. The coarse panel is sampled more sparsely so its lower-resolution structure remains visually distinct from the fine and predicted panels.

In [ ]:
# Plot a meteorological case study using the same saved prediction index.
plot_time_index = TIME_IDX
plot_time = np.asarray(saved_time)[plot_time_index]
if np.issubdtype(np.asarray(saved_time).dtype, np.datetime64):
    plot_time_label = np.datetime_as_string(plot_time, unit="h")
else:
    plot_time_label = str(plot_time)

fig = plot_case_study_comparison(
    coarse=saved_coarse[plot_time_index],
    fine=saved_fine[plot_time_index],
    predicted=saved_predicted[plot_time_index],
    lat=saved_lat,
    lon=saved_lon,
    variable_names=saved_variables,
    time_label=plot_time_label,
)

CASE_STUDY_PATH = f"{cfg.OUTPUT_DIR}/unet_case_study_comparison.png"
fig.savefig(CASE_STUDY_PATH, dpi=300, bbox_inches="tight")
display(fig)
plt.close(fig)

print("Saved case-study figure:", CASE_STUDY_PATH)
